# Stage 09 / Lecture 63: Structured Outputs via Pydantic (`BaseModel`, `Field`, `include_raw`, & Nested Schemas)

This production reference notebook enforces deterministic structured output contracts on LLM generations using **Pydantic v2** and `model.with_structured_output()`:
1. **Schema Binding (`Movie`)**: Contrasting unstructured prose outputs with strictly typed Pydantic instances.
2. **Runtime Field Validation**: Verifying `ValidationError` rejection on malformed schema data.
3. **Observability & Error Resilience (`include_raw=True`)**: Capturing `raw` `AIMessage` token telemetry alongside `parsed` structures and `parsing_error` states.
4. **Nested Hierarchical Schemas (`MovieDetails` + `Actor`)**: Extracting recursive typed lists (`List[Actor]`, `List[str]`, `Optional[float]`).


In [1]:
from typing import List, Optional, Dict, Any
import os
from dotenv import load_dotenv

from pydantic import BaseModel, Field, ValidationError
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY must be configured in .env for structured output tests.")

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("ChatGroq model and Pydantic v2 initialized successfully.")


ChatGroq model and Pydantic v2 initialized successfully.


### 1. Schema Definition, Field Validation, and Structured Output Binding

By inheriting from `pydantic.BaseModel` and annotating constraints via `Field(..., description=...)`, we restrict the LLM to populate typed attributes deterministically.


In [2]:
class Movie(BaseModel):
    title: str = Field(..., description="The exact title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie rating out of 10")


# 1. Verify runtime Pydantic validation catches malformed types
try:
    Movie(title="Inception", year="not_an_integer", director="Nolan", rating=8.8)
    raise AssertionError("Expected ValidationError for non-integer year.")
except ValidationError as exc:
    print(f"Verified strict Pydantic runtime validation ({len(exc.errors())} error caught).")

# 2. Unstructured Baseline
raw_prose = model.invoke("Provide details about the movie Inception in one concise plain-text sentence without emojis.")
assert isinstance(raw_prose, AIMessage)
print(f"Unstructured Baseline: {raw_prose.content}")

# 3. Structured Binding
structured_model = model.with_structured_output(Movie)
movie_obj: Movie = structured_model.invoke("Provide details about the movie Inception.")

assert isinstance(movie_obj, Movie)
assert movie_obj.title.lower() == "inception"
assert movie_obj.year == 2010
assert "nolan" in movie_obj.director.lower()
assert 0.0 < movie_obj.rating <= 10.0

print(f"Structured Output Object: {movie_obj}")
print(f"Extracted Attributes -> Title: {movie_obj.title} | Year: {movie_obj.year} | Rating: {movie_obj.rating}")


Verified strict Pydantic runtime validation (1 error caught).


Unstructured Baseline: Inception is a 2010 science‑fiction thriller directed by Christopher Nolan, starring Leonardo DiCaprio as a skilled thief who infiltrates people’s dreams to plant ideas.


Structured Output Object: title='Inception' year=2010 director='Christopher Nolan' rating=8.8
Extracted Attributes -> Title: Inception | Year: 2010 | Rating: 8.8


### 2. Preserving Token Telemetry and Fault Isolation (`include_raw=True`)

Passing `include_raw=True` returns a dictionary with `raw`, `parsed`, and `parsing_error`, preserving `usage_metadata` for production FinOps tracking and preventing sudden parsing exceptions from crashing pipelines.


In [3]:
structured_with_raw = model.with_structured_output(Movie, include_raw=True)
raw_bundle: Dict[str, Any] = structured_with_raw.invoke("Provide details about the movie Inception.")

assert set(raw_bundle.keys()) == {"raw", "parsed", "parsing_error"}
assert raw_bundle["parsing_error"] is None
assert isinstance(raw_bundle["raw"], AIMessage)
assert isinstance(raw_bundle["parsed"], Movie)

usage = raw_bundle["raw"].usage_metadata
assert usage is not None and usage["total_tokens"] > 0

print("include_raw=True Keys:", list(raw_bundle.keys()))
print("Parsed Instance:", raw_bundle["parsed"])
print("Parsing Error:", raw_bundle["parsing_error"])
print("Preserved Token Usage Telemetry:", usage)


include_raw=True Keys: ['raw', 'parsed', 'parsing_error']
Parsed Instance: title='Inception' year=2010 director='Christopher Nolan' rating=8.8
Parsing Error: None
Preserved Token Usage Telemetry: {'input_tokens': 160, 'output_tokens': 121, 'total_tokens': 281, 'output_token_details': {'reasoning': 82}}


### 3. Nested Hierarchical Schemas (`Actor` inside `MovieDetails`)

Complex enterprise extraction workflows require nesting child Pydantic models inside parent list attributes (`cast: List[Actor]`).


In [4]:
class Actor(BaseModel):
    name: str = Field(..., description="Full name of the actor")
    role: str = Field(..., description="Character name played in the movie")


class MovieDetails(BaseModel):
    title: str = Field(..., description="Movie title")
    year: int = Field(..., description="Release year")
    cast: List[Actor] = Field(..., description="List of top 3 leading cast members")
    genres: List[str] = Field(..., description="List of movie genres")
    budget_millions_usd: Optional[float] = Field(
        None,
        description="Production budget expressed in millions of USD (for example, write 160.0 for $160 million)"
    )


nested_model = model.with_structured_output(MovieDetails)
details: MovieDetails = nested_model.invoke(
    "Provide details about the movie Inception including top 3 actors, genres, and budget in millions."
)

assert isinstance(details, MovieDetails)
assert details.title.lower() == "inception"
assert details.year == 2010
assert len(details.cast) >= 2
assert all(isinstance(actor, Actor) for actor in details.cast)
assert len(details.genres) >= 1
assert details.budget_millions_usd is not None and 100.0 <= details.budget_millions_usd <= 300.0

print(f"Title: {details.title} ({details.year})")
print(f"Genres: {details.genres}")
print(f"Budget: ${details.budget_millions_usd}M")
print("Cast:")
for actor in details.cast:
    print(f"  - {actor.name} as {actor.role}")

print("\nAll Stage 09 / Lecture 63 Pydantic structured output assertions passed.")


Title: Inception (2010)
Genres: ['Action', 'Adventure', 'Sci-Fi']
Budget: $160.0M
Cast:
  - Leonardo DiCaprio as Dom Cobb
  - Joseph Gordon-Levitt as Arthur
  - Elliot Page as Ariadne

All Stage 09 / Lecture 63 Pydantic structured output assertions passed.
